# SMAP Soil Moisture Drought Trigger for US Counties

**Author:** Frank Anyoka Adekilae, Louisiana State University

This notebook builds a simple soil-moisture-based drought early-warning trigger for counties in the contiguous United States and tests how well it anticipates drought recorded by the US Drought Monitor (USDM).

**Workflow**
1. Clean daily SMAP 9 km soil moisture (one CSV per year) and save it as Parquet.
2. Assign each SMAP pixel to a county.
3. Average soil moisture to county-daily values.
4. Compute seasonal soil moisture anomalies (z-scores) against a day-of-year climatology.
5. Flag a dryness trigger when a county is persistently dry.
6. Compare the trigger with future USDM drought (D1 or worse) at 7–60 day lead times.

See the README for data sources and notes on methods.

## 1. Setup

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score

In [ ]:
# ---- Configuration: edit these paths for your setup -------------------------
DATA_DIR = "data"          # folder with SMAP_9km_pixels_<year>.csv and the USDM CSV
OUT_DIR  = "outputs"       # all processed files and figures are written here
YEARS    = range(2016, 2021)

SMAP_CSV_PATTERN = os.path.join(DATA_DIR, "SMAP_9km_pixels_{year}.csv")
USDM_CSV         = os.path.join(DATA_DIR, "USDM_county_2016_2020.csv")
COUNTIES_SOURCE  = "https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_county_5m.zip"

# Method settings
SM_MIN, SM_MAX   = 0.0, 0.7    # plausible volumetric soil moisture range (m3/m3)
CLIM_HALF_WINDOW = 15          # +/- days pooled around each day of year for the climatology
Z_THRESH         = -1.0        # a day is "dry" when z <= this value
TRIGGER_WINDOW   = "5D"        # calendar-day window for the trigger
TRIGGER_MIN_DAYS = 3           # dry days needed in the window to trigger
DSCI_EVENT       = 200         # USDM DSCI threshold used to define a drought event (see README)
LEAD_DAYS        = [7, 14, 21, 30, 45, 60]
MATCH_TOLERANCE  = "3D"        # max gap when matching a USDM date to the latest SMAP day

CONUS_EXCLUDE = {"02", "15", "72"}   # Alaska, Hawaii, Puerto Rico
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Optional: run in Google Colab with data on Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = "/content/drive/MyDrive/SMAP_data"
# OUT_DIR  = "/content/drive/MyDrive/SMAP_outputs"

## 2. Clean yearly SMAP files

Each CSV (exported from Google Earth Engine) has one row per pixel per day with `date`, `lat`, `lon`, and `sm`. We keep plausible values, round the coordinates so they work as a pixel key, and save compact Parquet files.

In [ ]:
def preprocess_smap_year(year):
    df = pd.read_csv(SMAP_CSV_PATTERN.format(year=year),
                     usecols=lambda c: c in {"date", "lat", "lon", "sm"})
    df["date"] = pd.to_datetime(df["date"])
    df = df[df["sm"].between(SM_MIN, SM_MAX)].dropna(subset=["sm"])
    # Stable pixel ID from rounded coordinates (safer than joining on floats)
    # Rounded coordinates act as a stable pixel key (safer than joining on raw floats)
    df["lat"] = df["lat"].round(4).astype("float32")
    df["lon"] = df["lon"].round(4).astype("float32")
    df["sm"] = df["sm"].astype("float32")
    out = os.path.join(OUT_DIR, f"smap_{year}.parquet")
    df.to_parquet(out, index=False)
    print(f"{year}: {len(df):,} rows -> {out}")

for year in YEARS:
    preprocess_smap_year(year)

## 3. Assign SMAP pixels to counties

Pixels are matched to the county that contains the pixel center. Counties smaller than a 9 km pixel may receive no pixel (see README).

In [ ]:
counties = gpd.read_file(COUNTIES_SOURCE)[["GEOID", "STATEFP", "NAME", "geometry"]]
counties = counties[~counties["STATEFP"].isin(CONUS_EXCLUDE)].to_crs("EPSG:4326")

# Unique pixels across ALL years
pixels = pd.concat(
    [pd.read_parquet(os.path.join(OUT_DIR, f"smap_{y}.parquet"), columns=["lat", "lon"]).drop_duplicates()
     for y in YEARS]
).drop_duplicates()

pts = gpd.GeoDataFrame(pixels, geometry=gpd.points_from_xy(pixels["lon"], pixels["lat"]), crs="EPSG:4326")
joined = gpd.sjoin(pts, counties[["GEOID", "geometry"]], how="inner", predicate="within")
pixel_to_county = joined[["lat", "lon", "GEOID"]].rename(columns={"GEOID": "county_fips"})
pixel_to_county.to_parquet(os.path.join(OUT_DIR, "pixel_to_county.parquet"), index=False)
print(f"{len(pixel_to_county):,} pixels mapped to {pixel_to_county['county_fips'].nunique():,} counties")

## 4. County-daily soil moisture

In [ ]:
parts = []
for y in YEARS:
    df = pd.read_parquet(os.path.join(OUT_DIR, f"smap_{y}.parquet"), columns=["lat", "lon", "date", "sm"])
    df = df.merge(pixel_to_county, on=["lat", "lon"], how="inner")
    parts.append(df.groupby(["county_fips", "date"], as_index=False)["sm"].mean()
                   .rename(columns={"sm": "sm_mean"}))
county = pd.concat(parts, ignore_index=True).sort_values(["county_fips", "date"])
county.to_parquet(os.path.join(OUT_DIR, "county_daily_sm.parquet"), index=False)
print(county.shape)
county.head()

## 5. Seasonal anomalies (z-scores)

For each county and day of year, the climatology is the mean and standard deviation of all values within ±15 days across all years. This removes the normal seasonal cycle, so a low z-score means drier than usual *for that time of year*.

In [ ]:
def add_seasonal_zscore(df, half_window=CLIM_HALF_WINDOW):
    # Day-of-year climatology per county, pooling +/- half_window days (circular)
    df = df.copy()
    df["doy"] = df["date"].dt.dayofyear.clip(upper=365)
    g = df.groupby(["county_fips", "doy"])["sm_mean"]
    stats = pd.DataFrame({"n": g.count(), "s": g.sum(), "ss": (df["sm_mean"] ** 2).groupby([df["county_fips"], df["doy"]]).sum()})
    wide = {k: stats[k].unstack("doy").reindex(columns=range(1, 366)).fillna(0).values for k in ["n", "s", "ss"]}
    pooled = {k: sum(np.roll(v, shift, axis=1) for shift in range(-half_window, half_window + 1)) for k, v in wide.items()}
    n, s, ss = pooled["n"], pooled["s"], pooled["ss"]
    with np.errstate(invalid="ignore", divide="ignore"):
        clim = s / n
        sd = np.sqrt((ss - n * clim ** 2) / (n - 1))
    counties_idx = stats["n"].unstack("doy").index
    clim_df = pd.DataFrame({
        "county_fips": np.repeat(counties_idx.values, 365),
        "doy": np.tile(np.arange(1, 366), len(counties_idx)),
        "clim": clim.ravel(),
        "sd": sd.ravel(),
    })
    df = df.merge(clim_df, on=["county_fips", "doy"], how="left")
    df["sm_anom"] = df["sm_mean"] - df["clim"]
    df["sm_z"] = df["sm_anom"] / df["sd"]
    return df

county = add_seasonal_zscore(county)
county[["sm_mean", "sm_anom", "sm_z"]].describe()

## 6. Dryness trigger

A county triggers when at least 3 of the last 5 **calendar** days are dry (z ≤ −1). A time-based window is used because SMAP 9 km has missing days.

In [ ]:
def add_trigger(df):
    df = df.sort_values(["county_fips", "date"]).copy()
    df["dry"] = (df["sm_z"] <= Z_THRESH).astype(int)
    df["dry_count"] = (df.set_index("date")
                         .groupby("county_fips")["dry"]
                         .rolling(TRIGGER_WINDOW).sum()
                         .values)
    df["trigger"] = (df["dry_count"] >= TRIGGER_MIN_DAYS).astype(int)
    return df

county = add_trigger(county)
county.to_parquet(os.path.join(OUT_DIR, "county_daily_sm_anom_trigger.parquet"), index=False)
print("Share of county-days with trigger on:", round(county["trigger"].mean(), 3))

## 7. US Drought Monitor events

USDM county statistics are weekly. Here a drought event is `DSCI >= 200` (see README for what this means and a stricter alternative).

In [ ]:
usdm = pd.read_csv(USDM_CSV)
usdm.columns = usdm.columns.str.strip()
usdm["county_fips"] = usdm["FIPS"].astype(str).str.zfill(5)
usdm["date"] = pd.to_datetime(usdm["MapDate"].astype(str), format="%Y%m%d")
usdm["event"] = (usdm["DSCI"] >= DSCI_EVENT).astype(int)
usdm = usdm[usdm["county_fips"].isin(county["county_fips"].unique())][["county_fips", "date", "DSCI", "event"]]
usdm.head()

## 8. Match dates and build future-event labels

Each USDM map date is matched to the most recent SMAP day within 3 days. For each lead time L, the label is 1 if a drought event occurs **after** the forecast date and within L days. Labels are left empty when the window runs past the end of the record.

In [ ]:
matched = pd.merge_asof(
    usdm.sort_values("date"),
    county[["county_fips", "date", "sm_z", "trigger"]].sort_values("date"),
    on="date", by="county_fips", direction="backward",
    tolerance=pd.Timedelta(MATCH_TOLERANCE),
).dropna(subset=["sm_z"])

def add_future_labels(df, lead_days=LEAD_DAYS):
    out = []
    for _, g in df.sort_values("date").groupby("county_fips"):
        g = g.copy()
        dates, ev, last = g["date"].values, g["event"].values, g["date"].values.max()
        for L in lead_days:
            lab = np.full(len(g), np.nan)
            for i, t in enumerate(dates):
                end = t + np.timedelta64(L, "D")
                if end <= last:
                    lab[i] = float(ev[(dates > t) & (dates <= end)].any())
            g[f"event_next_{L}d"] = lab
        out.append(g)
    return pd.concat(out, ignore_index=True)

matched = add_future_labels(matched)
matched.head()

## 9. Skill of the trigger

- **ROC-AUC / PR-AUC** use the continuous score −z (drier = higher score).
- **Hit rate** and **false-alarm ratio** describe the yes/no trigger.
- **Onset only** keeps forecast dates when the county is *not* already in drought, so the test measures early warning rather than current conditions.

In [ ]:
def skill_table(df, onset_only=True):
    rows = []
    for L in LEAD_DAYS:
        d = df.dropna(subset=[f"event_next_{L}d"])
        if onset_only:
            d = d[d["event"] == 0]
        y = d[f"event_next_{L}d"].astype(int).values
        if len(y) == 0 or y.min() == y.max():
            continue
        trig = d["trigger"].values
        rows.append({
            "lead_days": L,
            "n": len(y),
            "event_rate": y.mean(),
            "ROC_AUC": roc_auc_score(y, -d["sm_z"].values),
            "PR_AUC": average_precision_score(y, -d["sm_z"].values),
            "hit_rate": ((trig == 1) & (y == 1)).sum() / max(1, (y == 1).sum()),
            "false_alarm_ratio": ((trig == 1) & (y == 0)).sum() / max(1, (trig == 1).sum()),
        })
    return pd.DataFrame(rows)

skill_onset = skill_table(matched, onset_only=True)
skill_all = skill_table(matched, onset_only=False)
skill_onset.to_csv(os.path.join(OUT_DIR, "skill_onset.csv"), index=False)
skill_all.to_csv(os.path.join(OUT_DIR, "skill_all.csv"), index=False)
skill_onset

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(skill_onset["lead_days"], skill_onset["ROC_AUC"], marker="o", label="ROC-AUC")
ax.plot(skill_onset["lead_days"], skill_onset["PR_AUC"], marker="s", label="PR-AUC")
ax.plot(skill_onset["lead_days"], skill_onset["event_rate"], ls="--", marker="x", label="PR no-skill (event rate)")
ax.axhline(0.5, color="grey", lw=0.8, ls=":", label="ROC no-skill")
ax.set_xlabel("Lead time (days)")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.set_title("SMAP dryness score vs. future USDM drought onset")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "skill_vs_lead_time.png"), dpi=300)
plt.show()

## 10. Example: one county time series

In [ ]:
def plot_county(fips):
    c = county[county["county_fips"] == fips]
    if c.empty:
        print("No data for", fips)
        return
    fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
    axes[0].plot(c["date"], c["sm_mean"], lw=0.8)
    axes[0].set_ylabel("SM (m³/m³)")
    axes[1].plot(c["date"], c["sm_z"], lw=0.8)
    axes[1].axhline(0, color="black", lw=0.8)
    axes[1].fill_between(c["date"], c["sm_z"].min(), c["sm_z"].max(),
                         where=c["trigger"] == 1, alpha=0.2, label="Trigger on")
    axes[1].set_ylabel("SM z-score")
    axes[1].legend(loc="lower left")
    axes[0].set_title(f"County {fips}: SMAP soil moisture and seasonal anomaly")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"county_{fips}_timeseries.png"), dpi=300)
    plt.show()

plot_county(county["county_fips"].iloc[0])   # replace with a FIPS code, e.g. "22033" (East Baton Rouge)